# Improved Satellite Image Threat Detection

This notebook converts xView-style GeoTIFF imagery and GeoJSON labels into YOLO chips, trains YOLOv8, validates the model, and includes tiled inference for large satellite images.


In [ ]:
# !rm -rf /kaggle/working/*

In [ ]:
!pip install ultralytics rasterio geopandas albumentations torchvision shapely tqdm pyyaml

## Imports and Configuration


In [ ]:
from __future__ import annotations

import json
import os
import random
from collections import Counter
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import rasterio
import torch
import torchvision
import yaml
from rasterio.windows import Window
from shapely.geometry import box
from tqdm import tqdm
from ultralytics import YOLO


SEED = 42
CHIP_SIZE = 512 #800
STRIDE = 364 #640
VISIBILITY_THRESHOLD = 0.4
VAL_FRACTION = 0.2
BACKGROUND_KEEP_PROB = 0.08
MODEL_WEIGHTS = "yolov8m.pt"


## Reproducibility and Paths


In [ ]:
def seed_everything(seed: int = SEED) -> None:
    """
    Seeds all major random number generators to ensure reproducibility.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
def get_paths() -> tuple[Path, Path, Path]:
    """
    This function dynamically locates and returns key file paths for the xView 
    dataset based on whether the script is running in Kaggle or a local environment.

    How it works:
    1. Environment Check:
       - Looks for a "/kaggle" directory to see if it is running on the Kaggle platform.
    
    2. Path Selection:
       - If on Kaggle: Sets the root to the input dataset location and the output 
         directory to Kaggle's writable working folder.
       - If Local: Resolves relative paths to a "./data" folder and an "./xview_yolo" 
         output folder on your own machine.
    
    3. File Construction:
       - Combines the chosen root path with specific subdirectories to target the 
         training images folder and the GeoJSON format label file.
    
    4. Return Values:
       - Returns a 3-element tuple containing the final Image Directory, Label Path, 
         and Output Directory.
    """
    
    if Path("/kaggle").exists():
        root = Path("/kaggle/input/datasets/hassanmojab/xview-dataset")
        output_dir = Path("/kaggle/working/xview_yolo")
    else:
        root = Path("./data").resolve()
        output_dir = Path("./xview_yolo").resolve()

    image_dir = root / "train_images" / "train_images"
    label_path = root / "train_labels" / "xView_train.geojson"
    return image_dir, label_path, output_dir


In [ ]:
def print_device_info() -> None:
    """
    This function is for checking if CUDA is available
    - Then GPU
    Else
    - CPU
    """
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU device: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU device: CPU only")


In [ ]:
def prepare_dirs(output_dir: Path) -> None:
    for split in ("train", "val"):
        (output_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (output_dir / "labels" / split).mkdir(parents=True, exist_ok=True)


## Annotation Loading and Class Mapping


In [ ]:
def load_annotations(label_path: Path) -> pd.DataFrame:
    if not label_path.exists():
        raise FileNotFoundError(f"Label file not found: {label_path}")

    with label_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for feature in data["features"]:
        props = feature["properties"]
        coords = props.get("bounds_imcoords", "")
        if not coords:
            continue

        x1, y1, x2, y2 = map(int, coords.split(","))
        if x2 <= x1 or y2 <= y1:
            continue

        rows.append(
            {
                "image_id": props["image_id"],
                "type_id": int(props["type_id"]),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
            }
        )

    ann_df = pd.DataFrame(rows)
    if ann_df.empty:
        raise ValueError("No valid annotations were found.")

    return ann_df


In [ ]:
def filter_existing_images(ann_df: pd.DataFrame, image_dir: Path) -> pd.DataFrame:
    existing_images = {image_path.name for image_path in image_dir.glob("*.tif")}
    before_annotations = len(ann_df)
    before_images = ann_df["image_id"].nunique()

    filtered_df = ann_df[ann_df["image_id"].isin(existing_images)].copy()
    removed_annotations = before_annotations - len(filtered_df)
    removed_images = before_images - filtered_df["image_id"].nunique()

    print(
        "After removing missing images: "
        f"{len(filtered_df):,} annotations from {filtered_df['image_id'].nunique():,} images."
    )
    if removed_images:
        print(
            f"Removed {removed_annotations:,} annotations linked to "
            f"{removed_images:,} missing images."
        )

    if filtered_df.empty:
        raise ValueError(f"No annotated images were found in: {image_dir}")

    return filtered_df


In [ ]:
def build_class_mapping(ann_df: pd.DataFrame) -> dict[int, int]:
    type_ids = sorted(int(type_id) for type_id in ann_df["type_id"].unique())
    return {int(type_id): int(idx) for idx, type_id in enumerate(type_ids)}


In [ ]:
def split_images(ann_df: pd.DataFrame, val_fraction: float = VAL_FRACTION) -> dict[str, str]:
    image_ids = sorted(ann_df["image_id"].unique())
    rng = random.Random(SEED)
    rng.shuffle(image_ids)

    val_count = max(1, int(len(image_ids) * val_fraction))
    val_images = set(image_ids[:val_count])

    return {image_id: ("val" if image_id in val_images else "train") for image_id in image_ids}


## Chipping Helpers


In [ ]:
def calculate_visibility(box_coords: tuple[int, int, int, int], chip_box) -> float:
    obj_box = box(*box_coords)
    if obj_box.area <= 0:
        return 0.0

    intersection = obj_box.intersection(chip_box)
    if intersection.is_empty:
        return 0.0

    return float(intersection.area / obj_box.area)


In [ ]:
def tile_starts(length: int, tile_size: int, stride: int) -> list[int]:
    if length <= tile_size:
        return [0]

    starts = list(range(0, length - tile_size + 1, stride))
    final_start = length - tile_size
    if starts[-1] != final_start:
        starts.append(final_start)
    return starts


In [ ]:
def normalize_to_uint8(chip: np.ndarray) -> np.ndarray:
    chip = np.asarray(chip)
    if chip.dtype == np.uint8:
        return chip

    chip = chip.astype(np.float32)
    lo, hi = np.percentile(chip, (1, 99))
    if hi <= lo:
        return np.zeros_like(chip, dtype=np.uint8)
    chip = np.clip((chip - lo) * 255.0 / (hi - lo), 0, 255)
    return chip.astype(np.uint8)


## Augmentation and Saving


In [ ]:
def make_transform() -> A.Compose:
    return A.Compose(
        [
            A.CLAHE(p=0.35),
            A.RandomRotate90(p=0.5),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.35),
        ],
        bbox_params=A.BboxParams(
            format="yolo",
            label_fields=["class_labels"],
            min_visibility=0.2,
        ),
    )


In [ ]:
def save_chip(
    chip: np.ndarray,
    boxes_yolo: list[list[float]],
    output_dir: Path,
    split: str,
    chip_name: str,
) -> None:
    image_path = output_dir / "images" / split / f"{chip_name}.jpg"
    label_path = output_dir / "labels" / split / f"{chip_name}.txt"

    cv2.imwrite(str(image_path), cv2.cvtColor(chip, cv2.COLOR_RGB2BGR))
    with label_path.open("w", encoding="utf-8") as f:
        for cls, x_center, y_center, width, height in boxes_yolo:
            f.write(f"{int(cls)} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")


## YOLO Chip Generation


In [ ]:
def create_yolo_chips(
    image_dir: Path,
    ann_df: pd.DataFrame,
    output_dir: Path,
    class_mapping: dict[int, int],
    image_split: dict[str, str],
    use_augmentation: bool = True,
) -> Counter:
    transform = make_transform()
    chip_count = 0
    split_counts: Counter = Counter()

    for image_name in tqdm(sorted(ann_df["image_id"].unique()), desc="Creating chips"):
        image_path = image_dir / image_name
        if not image_path.exists():
            print(f"Skipping missing image: {image_path}")
            continue

        split = image_split[image_name]
        image_annotations = ann_df[ann_df["image_id"] == image_name]

        with rasterio.open(image_path) as src:
            width, height = src.width, src.height
            x_starts = tile_starts(width, CHIP_SIZE, STRIDE)
            y_starts = tile_starts(height, CHIP_SIZE, STRIDE)

            for y in y_starts:
                for x in x_starts:
                    window = Window(x, y, CHIP_SIZE, CHIP_SIZE)
                    chip = src.read(window=window)
                    if chip.shape[1] != CHIP_SIZE or chip.shape[2] != CHIP_SIZE:
                        continue

                    chip = np.transpose(chip[:3], (1, 2, 0))
                    chip = normalize_to_uint8(chip)
                    chip_polygon = box(x, y, x + CHIP_SIZE, y + CHIP_SIZE)
                    boxes_yolo = []

                    for row in image_annotations.itertuples(index=False):
                        visibility = calculate_visibility((row.x1, row.y1, row.x2, row.y2), chip_polygon)
                        if visibility < VISIBILITY_THRESHOLD:
                            continue

                        nx1 = max(0, row.x1 - x)
                        ny1 = max(0, row.y1 - y)
                        nx2 = min(CHIP_SIZE, row.x2 - x)
                        ny2 = min(CHIP_SIZE, row.y2 - y)

                        box_width = nx2 - nx1
                        box_height = ny2 - ny1
                        if box_width <= 1 or box_height <= 1:
                            continue

                        boxes_yolo.append(
                            [
                                class_mapping[row.type_id],
                                ((nx1 + nx2) / 2.0) / CHIP_SIZE,
                                ((ny1 + ny2) / 2.0) / CHIP_SIZE,
                                box_width / CHIP_SIZE,
                                box_height / CHIP_SIZE,
                            ]
                        )

                    keep_background = not boxes_yolo and random.random() < BACKGROUND_KEEP_PROB
                    if not boxes_yolo and not keep_background:
                        continue

                    if use_augmentation and split == "train" and boxes_yolo:
                        class_labels = [int(b[0]) for b in boxes_yolo]
                        bboxes = [b[1:] for b in boxes_yolo]
                        augmented = transform(image=chip, bboxes=bboxes, class_labels=class_labels)
                        chip = augmented["image"]
                        boxes_yolo = [
                            [cls, *bbox]
                            for cls, bbox in zip(augmented["class_labels"], augmented["bboxes"])
                        ]
                        if not boxes_yolo:
                            continue

                    chip_name = f"{Path(image_name).stem}_{x}_{y}_{chip_count}"
                    save_chip(chip, boxes_yolo, output_dir, split, chip_name)
                    chip_count += 1
                    split_counts[split] += 1

    return split_counts


## Dataset YAML


In [ ]:
def write_data_yaml(output_dir: Path, class_mapping: dict[int, int]) -> Path:
    clean_mapping = {int(type_id): int(idx) for type_id, idx in class_mapping.items()}
    names = {idx: f"xview_type_{type_id}" for type_id, idx in clean_mapping.items()}
    data_yaml = {
        "path": str(output_dir),
        "train": "images/train",
        "val": "images/val",
        "nc": len(names),
        "names": names,
    }

    yaml_path = output_dir / "data.yaml"
    with yaml_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)

    mapping_path = output_dir / "class_mapping.json"
    with mapping_path.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "type_id_to_yolo_index": clean_mapping,
                "yolo_index_to_type_id": {idx: type_id for type_id, idx in clean_mapping.items()},
            },
            f,
            indent=2,
        )

    return yaml_path


## Training and Validation


In [ ]:
def train_model(data_yaml: Path, output_dir: Path) -> YOLO:
    model = YOLO(MODEL_WEIGHTS)
    model.train(
        data=str(data_yaml),
        epochs=50,
        imgsz=CHIP_SIZE,
        batch=8,
        workers=2,
        cache=False,
        optimizer="auto",
        mosaic=1.0,
        copy_paste=0.2,
        degrees=90,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        scale=0.5,
        translate=0.1,
        project=str(output_dir),
        name="satellite_detector",
        seed=SEED,
    )
    return model


In [ ]:
def validate_model(model: YOLO) -> None:
    metrics = model.val()
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")


## Large Image Inference


In [ ]:
def predict_large_image(
    image_path: str | Path,
    model_path: str | Path,
    tile_size: int = CHIP_SIZE,
    overlap: int = 100,
    conf_threshold: float = 0.25,
    iou_threshold: float = 0.45,
) -> list[list[float]]:
    model = YOLO(str(model_path))
    stride = tile_size - overlap
    if stride <= 0:
        raise ValueError("overlap must be smaller than tile_size")

    all_boxes = []
    all_scores = []
    all_classes = []

    with rasterio.open(image_path) as src:
        width, height = src.width, src.height
        for y in tile_starts(height, tile_size, stride):
            for x in tile_starts(width, tile_size, stride):
                window = Window(x, y, tile_size, tile_size)
                img = src.read(window=window)
                if img.shape[1] != tile_size or img.shape[2] != tile_size:
                    continue

                img = np.transpose(img[:3], (1, 2, 0))
                img = normalize_to_uint8(img)
                results = model.predict(img, imgsz=tile_size, conf=conf_threshold, verbose=False)

                for result in results:
                    for pred_box in result.boxes:
                        xyxy = pred_box.xyxy[0].cpu().numpy()
                        conf = float(pred_box.conf[0].cpu().numpy())
                        cls = int(pred_box.cls[0].cpu().numpy())

                        all_boxes.append(
                            [
                                float(xyxy[0] + x),
                                float(xyxy[1] + y),
                                float(xyxy[2] + x),
                                float(xyxy[3] + y),
                            ]
                        )
                        all_scores.append(conf)
                        all_classes.append(cls)

    if not all_boxes:
        return []

    keep = torchvision.ops.batched_nms(
        boxes=torch.tensor(all_boxes, dtype=torch.float32),
        scores=torch.tensor(all_scores, dtype=torch.float32),
        idxs=torch.tensor(all_classes, dtype=torch.int64),
        iou_threshold=iou_threshold,
    )

    return [
        [
            *all_boxes[i],
            float(all_scores[i]),
            int(all_classes[i]),
        ]
        for i in keep.tolist()
    ]


## Run Full Pipeline

Run this cell after confirming the dataset paths. It creates chips, writes `data.yaml`, trains YOLOv8, and validates the best run.


In [ ]:
seed_everything()
print_device_info()

image_dir, label_path, output_dir = get_paths()
print(f"Image directory: {image_dir}")
print(f"Label path: {label_path}")
print(f"Output directory: {output_dir}")

prepare_dirs(output_dir)
ann_df = load_annotations(label_path)
print(f"Loaded {len(ann_df):,} valid annotations from {ann_df['image_id'].nunique():,} images.")
ann_df = filter_existing_images(ann_df, image_dir)

class_mapping = build_class_mapping(ann_df)
image_split = split_images(ann_df)
data_yaml = write_data_yaml(output_dir, class_mapping)

split_counts = create_yolo_chips(
    image_dir=image_dir,
    ann_df=ann_df,
    output_dir=output_dir,
    class_mapping=class_mapping,
    image_split=image_split,
    use_augmentation=True,
)
print(f"Chip counts: {dict(split_counts)}")
print(f"Data YAML saved at: {data_yaml}")

model = train_model(data_yaml, output_dir)
validate_model(model)

best_model = output_dir / "satellite_detector" / "weights" / "best.pt"
print(f"Best model expected at: {best_model}")


## Example Large Image Prediction

Update `sample_image` and `trained_model` after training, then run this cell.


In [ ]:
# image_dir="/kaggle/input/datasets/hassanmojab/xview-dataset/val_images/val_images"
# sample_image = image_dir / '1038.tif'
# trained_model = output_dir / 'satellite_detector' / 'weights' / 'best.pt'

# if sample_image.exists() and trained_model.exists():
#     global_boxes = predict_large_image(sample_image, trained_model, iou_threshold=0.45)
#     print(f"First 5 predictions: {global_boxes[:5]}")
# else:
#     print('Train the model first or update the sample image/model paths.')


In [ ]:
from pathlib import Path

# Convert strings to Path objects
image_dir = Path("/kaggle/input/datasets/hassanmojab/xview-dataset/val_images/val_images")
output_dir = Path("/kaggle/working/xview_yolo") # Ensure this is also a Path object

# Now this will work:
sample_image = image_dir / '1038.tif'
trained_model = output_dir / 'satellite_detector' / 'weights' / 'best.pt'

In [ ]:
if sample_image.exists() and trained_model.exists():
    global_boxes = predict_large_image(sample_image, trained_model, iou_threshold=0.45)
    print(f"First 5 predictions: {global_boxes[:5]}")
else:
    print('Train the model first or update the sample image/model paths.')


In [ ]:
result= pd.read_csv("/kaggle/working/xview_yolo/satellite_detector/results.csv")

In [ ]:
def inspect_per_class_performance(model: YOLO):
    # 1. Run validation
    metrics = model.val()
    
    # 2. Access the class-wise results
    # The 'class_result' attribute contains a list/array of metrics for each class
    # We iterate through them to print the names and their respective mAP50
    
    print(f"{'Class ID':<10} | {'Class Name':<20} | {'mAP50':<10}")
    print("-" * 45)
    
    for i, class_name in enumerate(model.names.values()):
        # Extract mAP50 for this specific class
        # metrics.box.ap50 is an array where each index corresponds to a class
        class_map50 = metrics.box.ap50[i]
        
        print(f"{i:<10} | {class_name:<20} | {class_map50:.4f}")

# Call this function after your model has finished training
# inspect_per_class_performance(model)